In [ ]:
import cv2
import os
import glob
import numpy as np
from wisardpkg import ClusWisard
import sys
sys.path.append('/mnt/c/Users/Isabella/tcc')
from wisard_object_tracker.src.utils import tracker_utils
import matplotlib.pyplot as plt

DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger2_gt.txt')

In [ ]:
# Carrega imagens
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# Carrega ground truths
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

### Tratamento de fundo + Otsu

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def preprocess_frame(frame):
    # Converte para grayscale caso seja RGB
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()
    # piorou apenas com otsu sem média global
    # 1) MÉDIA GLOBAL  (correção do erro)
    global_mean = int(np.mean(gray))     # precisa ser escalar Python
    no_bg_global = cv2.subtract(gray, global_mean)

    # 2) Binarização Otsu
    _, otsu = cv2.threshold(
        no_bg_global, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    # 3) Retornar vetor flatten binário
    vector = (otsu.flatten() > 0).astype(np.uint8)

    return vector


## Analise de Sensibilidade

Aqui eu estou treinando com um patch e classificando os 10 próximos para avaliar a sensibilidade da ativação

In [ ]:
# --- PASSO 3: INSTANCIANDO CLUSWISARD ---

# Parâmetros simples
parameters = {
    "INPUT_PATTERN_SIDE": 32,                  
    "CLUSWISARD_ADDRESS_SIZE": 6,          
    "ACCEPTABLE_ACTIVATION_SCORE": 0.5,       
    "MIN_ACCEPTABLE_ACTIVATION_SCORE": 0.1,    
    "CLUSWISARD_MIN_SCORE": 0.15,               
    "CLUSWISARD_THRESHOLD": 20,                 
    "CLUSWISARD_DISCRIMINATOR_LIMIT": 20,      
    "CLUSWISARD_BLEACHING_ACTIVATED": True,     
    "CLUSWISARD_ACTIVATION_DEGREE": True,
    "CLUSWISARD_RETURN_CONFIDENCE": True,
    "CLUSWISARD_CLASSES_DEGREES": True,
    "INITIAL_FRAME": 1,
    "LAST_FRAME": 25,
    "MAX_SEARCH_WINDOW_SCALE": 2,
}

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

PATCH_SIZE = (64, 64)   # tamanho fixo usado no treino e teste
STEP = 5                # pular de 5 em 5 frames
TEST_WINDOW = 10        # testar nos próximos 10 frames

all_results = []        # para salvar as ativações

for i in range(0, len(image_paths), STEP):

    print(f"\n=== BASE {i} — {image_paths[i]} ===")

    # -----------------------------
    # 1) CRIAR NOVO MODELO
    # -----------------------------
    model = ClusWisard(
        parameters['CLUSWISARD_ADDRESS_SIZE'],
        parameters['CLUSWISARD_MIN_SCORE'],
        parameters['CLUSWISARD_THRESHOLD'],
        parameters['CLUSWISARD_DISCRIMINATOR_LIMIT'],
        bleachingActivated=parameters['CLUSWISARD_BLEACHING_ACTIVATED'],
        returnActivationDegree=parameters['CLUSWISARD_ACTIVATION_DEGREE'],
        returnConfidence=parameters['CLUSWISARD_RETURN_CONFIDENCE'],
        returnClassesDegrees=parameters['CLUSWISARD_CLASSES_DEGREES']
    )

    # -----------------------------
    # 2) TREINO
    # -----------------------------
    frame = cv2.imread(image_paths[i], cv2.IMREAD_GRAYSCALE)
    patch = tracker_utils.extract_patch(frame, all_ground_truths[i])
    patch = cv2.resize(patch, PATCH_SIZE)

    pattern = preprocess_frame(patch).tolist()
    model.train([pattern], ["object"])

    # -----------------------------
    # 3) TESTE NOS PRÓXIMOS FRAMES
    # -----------------------------
    max_k = min(i + TEST_WINDOW + 1, len(image_paths))

    fig, axes = plt.subplots(1, max_k - i, figsize=(24, 3))
    plt.suptitle(f"Ativações — Base {i}", fontsize=14)

    for j, k in enumerate(range(i, max_k)):

        frame_k = cv2.imread(image_paths[k], cv2.IMREAD_GRAYSCALE)
        patch_k = tracker_utils.extract_patch(frame_k, all_ground_truths[i])
        patch_k = cv2.resize(patch_k, PATCH_SIZE)

        pattern_k = preprocess_frame(patch_k).tolist()

        # classificar
        result = model.classify([pattern_k])[0]
        activation = result.get("activationDegree", -1)

        # plot
        axes[j].imshow(patch_k, cmap="gray")
        axes[j].set_title(f"{k}\n{activation:.2f}")
        axes[j].axis("off")

        # salvar log
        all_results.append({
            "base": i,
            "test_frame": k,
            "activation": activation
        })

    plt.tight_layout()
    plt.show()


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

PATCH_SIZE = (64, 64)
STEP = 5

all_results = []

# -----------------------------
# 1) CRIAR MODELO
# -----------------------------
model = ClusWisard(
    parameters['CLUSWISARD_ADDRESS_SIZE'],
    parameters['CLUSWISARD_MIN_SCORE'],
    parameters['CLUSWISARD_THRESHOLD'],
    parameters['CLUSWISARD_DISCRIMINATOR_LIMIT'],
    bleachingActivated=parameters['CLUSWISARD_BLEACHING_ACTIVATED'],
    returnActivationDegree=parameters['CLUSWISARD_ACTIVATION_DEGREE'],
    returnConfidence=parameters['CLUSWISARD_RETURN_CONFIDENCE'],
    returnClassesDegrees=parameters['CLUSWISARD_CLASSES_DEGREES']
)

# -----------------------------
# 2) TREINO COM PATCH DO FRAME 0
# -----------------------------
frame_0 = cv2.imread(image_paths[0], cv2.IMREAD_GRAYSCALE)
patch_0 = tracker_utils.extract_patch(frame_0, all_ground_truths[0])
patch_0 = cv2.resize(patch_0, PATCH_SIZE)

pattern_0 = preprocess_frame(patch_0).tolist()
model.train([pattern_0], ["object"])

print("Modelo treinado com o patch do frame 0")

# -----------------------------
# 3) CLASSIFICAÇÃO DE 5 EM 5
# -----------------------------
for k in range(0, len(image_paths), STEP):

    frame_k = cv2.imread(image_paths[k], cv2.IMREAD_GRAYSCALE)

    # ⬇️ AGORA usa o ground truth do próprio frame k
    patch_k = tracker_utils.extract_patch(frame_k, all_ground_truths[k])
    patch_k = cv2.resize(patch_k, PATCH_SIZE)

    pattern_k = preprocess_frame(patch_k).tolist()

    # classificar
    result = model.classify([pattern_k])[0]
    activation = result.get("activationDegree", -1)

    # log no console
    print(f"Frame {k} — Activation: {activation:.4f}")

    # plot
    plt.figure(figsize=(3, 3))
    plt.imshow(patch_k, cmap="gray")
    plt.title(f"Frame {k}\nActivation: {activation:.2f}")
    plt.axis("off")
    plt.show()

    # salvar resultados
    all_results.append({
        "train_frame": 0,
        "test_frame": k,
        "activation": activation
    })


## Analise de ativação